In [ ]:
import numpy as np
from numba import cuda

In [ ]:
!uv pip install -q --system numba-cuda==0.4.0

In [ ]:
from numba import config
config.CUDA_ENABLE_PYNVJITLINK = 1
config.CUDA_LOW_OCCUPANCY_WARNINGS = 0

## Dot product

$$
\begin{bmatrix}
a_1,a_2,\dots,a_n
\end{bmatrix}
\times
\begin{bmatrix}
b_1,b_2,\dots,b_n
\end{bmatrix}
=
a_1b_1+a_2b_2+,\dots,+a_nb_n
$$

## `dot_vec` function without ufunc

In [ ]:
def dot_vec(va, vb):
    adim = va.size
    #bdim=vb.size ; bdim == adim
    res = np.zeros(adim)
    for i in range(adim):
        res[i] += va[i] * vb[i]
    dotprod = sum(res[:])
    return dotprod

## Define the kernel

In [ ]:
@cuda.jit
def g_dot_vec(va, vb, res):
    ti = cuda.grid(1)
    if ti < va.size:
        res[ti] = va[ti] * vb[ti]

## Host vector initialization

In [ ]:
va = np.random.randn(1024)
vb = np.random.randn(1024)
va = va.astype(np.float32)
vb = vb.astype(np.float32)

In [ ]:
dot_vec(va,vb)

## Copy and allocation on the device

In [ ]:
g_va = cuda.to_device(va)
g_vb = cuda.to_device(vb)
g_res = cuda.device_array_like(va)

## Call the kernel

In [ ]:
threadsperblock = 128
blockspergrid = (va.size + (threadsperblock - 1)) // threadsperblock

g_dot_vec[blockspergrid, threadsperblock](g_va, g_vb, g_res)

## Copy the result from device to host

In [ ]:
h_res = g_res.copy_to_host()

## Finalize the algorithm (sum) on the host

In [ ]:
dotprod = sum(h_res[:])

## Check the result

In [ ]:
print(dotprod - dot_vec(va,vb))

## Modify the kernel and the workflow to compute the sum on the device

In [ ]:
@cuda.jit
def complete_dot_vec(va, vb, res):
    # Shared memory for partial sums within a block
    sdata = cuda.shared.array(128, dtype=cuda.float32)

    # Global and local thread indices
    tid = cuda.threadIdx.x
    i = cuda.grid(1)

    # Each thread computes its partial product
    tmp = 0.0
    if i < va.size:
        tmp = va[i] * vb[i]

    # Store in shared memory
    sdata[tid] = tmp
    cuda.syncthreads()

    # Reduction in shared memory
    block_size = cuda.blockDim.x
    stride = block_size // 2
    while stride > 0:
        if tid < stride:
            sdata[tid] += sdata[tid + stride]
        cuda.syncthreads()
        stride //= 2

    # Write block result to global memory
    if tid == 0:
        res[cuda.blockIdx.x] = sdata[0]

## Host vector initialization

In [ ]:
va = np.random.randn(1024)
vb = np.random.randn(1024)
va = va.astype(np.float32)
vb = vb.astype(np.float32)

## Copy and allocation on the device

In [ ]:
g_va = cuda.to_device(va)
g_vb = cuda.to_device(vb)
g_res = cuda.device_array(blockspergrid, dtype=np.float32)

## Call the kernel

In [ ]:
threadsperblock = 128
blockspergrid = (va.size + (threadsperblock - 1)) // threadsperblock

g_dot_vec[blockspergrid, threadsperblock](g_va, g_vb, g_res)

In [ ]:
va = np.random.randn(N).astype(np.float32)
vb = np.random.randn(N).astype(np.float32)

# Move data to GPU
g_va = cuda.to_device(va)
g_vb = cuda.to_device(vb)
g_partial = cuda.device_array(blockspergrid, dtype=np.float32)

# Launch first reduction kernel
g_dot_vec[blockspergrid, threadsperblock](g_va, g_vb, g_partial)

# If multiple blocks, we need to reduce again
while blockspergrid > 1:
    prev_blocks = blockspergrid
    blockspergrid = (prev_blocks + threadsperblock - 1) // threadsperblock
    g_partial_next = cuda.device_array(blockspergrid, dtype=np.float32)
    g_dot_vec[blockspergrid, threadsperblock](g_partial, g_partial, g_partial_next)
    g_partial = g_partial_next



## Copy the result from device to host

In [ ]:
dotprod = g_res.copy_to_host()[0]

## Check the result

In [ ]:
cpu_dot = np.dot(va, vb)
print("GPU dot product:", dotprod)
print("CPU dot product:", cpu_dot)
print("Difference:", abs(dotprod - cpu_dot))